# Robosuite: Train SmolVLA + Evaluate Nut Assembly (with Video)

End-to-end pipeline on Google Colab:
1. **Collect** scripted expert demonstrations from 6 robosuite environments
2. **Convert** demos to LeRobot v3.0 dataset format
3. **Train** SmolVLA (450M params) on the multi-task dataset
4. **Evaluate** on nut assembly tasks, recording **MP4 video of every episode**

**Runtime:** A100 GPU (Runtime > Change runtime type > A100)

**Time:** ~25 min collection + ~15 min conversion + ~5-6 hrs training + ~30 min eval

**Tip:** Save checkpoints to Google Drive (Section 8) to survive session disconnects.

## 1. System Setup & Dependencies

Install MuJoCo rendering libs, robosuite, LeRobot with SmolVLA support.

In [ ]:
%%bash
# System dependencies for headless MuJoCo rendering
apt-get update -qq
apt-get install -y -qq libegl1-mesa-dev libgl1-mesa-glx libosmesa6-dev libglfw3 ffmpeg patchelf > /dev/null 2>&1

# NVIDIA EGL ICD config (Colab is missing this by default)
mkdir -p /usr/share/glvnd/egl_vendor.d
cat > /usr/share/glvnd/egl_vendor.d/10_nvidia.json << 'EOF'
{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}
EOF

# Python dependencies
pip install -q robosuite imageio[ffmpeg] h5py matplotlib seaborn pandas tqdm rich

# LeRobot + SmolVLA
if [ ! -d "lerobot" ]; then
    git clone https://github.com/huggingface/lerobot.git
fi
cd lerobot && pip install -q -e ".[smolvla]"

# Pin numpy for compatibility (Colab needs >=2.0, numba needs <2.1)
pip install -q "numpy>=2.0,<2.1"

echo "Done. Restart runtime now (next cell)."

In [ ]:
# Restart runtime to clear stale numpy C bindings
# After restart, SKIP the install cell above and run the next cell
import os
os.kill(os.getpid(), 9)

In [ ]:
# Post-restart setup — run this cell after restart
import os

# MUST be set BEFORE importing mujoco or robosuite
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"

# Suppress noisy TensorFlow/CUDA warnings that don't affect training
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import robosuite
macros_private = os.path.join(os.path.dirname(robosuite.__file__), "macros_private.py")
if not os.path.exists(macros_private):
    with open(macros_private, "w") as f:
        f.write("# Auto-generated\n")

import numpy as np
import torch

print(f"robosuite: {robosuite.__version__}")
print(f"numpy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import lerobot
print(f"LeRobot: {lerobot.__version__}")
print("Ready!")

## 2. Clone Project Repository

In [ ]:
%%bash
if [ ! -d "AUTOLAB-Project" ]; then
    git clone https://github.com/williampacini/AUTOLAB-Project.git
fi
cd AUTOLAB-Project && git fetch origin claude/debug-streaming-output-YBXGD && git checkout claude/debug-streaming-output-YBXGD && git pull origin claude/debug-streaming-output-YBXGD
echo "Repo ready (branch: claude/debug-streaming-output-YBXGD)"
ls robosuite-vla/

In [ ]:
# Change working directory to robosuite-vla
import os
os.chdir("AUTOLAB-Project/robosuite-vla")
print(f"Working directory: {os.getcwd()}")

## 3. Collect Scripted Expert Demos

Scripted policies use ground-truth object positions to generate perfect demonstrations.
50 demos per task (75 for NutAssembly) = 325 total demos.

In [ ]:
# Configuration
N_DEMOS = 50          # demos per environment
NUT_DEMOS = 75        # extra demos for the harder NutAssembly task

# For quick testing, uncomment:
# N_DEMOS = 2
# NUT_DEMOS = 2

In [ ]:
# Collect demos for all environments
import subprocess, sys

envs = [
    ("Lift", N_DEMOS),
    ("Stack", N_DEMOS),
    ("PickPlaceSingle", N_DEMOS),
    ("Door", N_DEMOS),
    ("NutAssemblySingle", N_DEMOS),
    ("NutAssembly", NUT_DEMOS),
]

for env_name, n_demos in envs:
    print(f"\n{'='*50}")
    print(f"  Collecting {n_demos} demos for {env_name}")
    print(f"{'='*50}")
    !python data/collect_demos.py --env {env_name} --n-demos {n_demos}

# Verify
print("\n" + "="*50)
print("  Collection complete")
print("="*50)
!ls -lh data/robosuite_demos/*.hdf5 2>/dev/null || echo "No HDF5 files found"

## 3b. Preview Training Demos (Video)

Render sample demonstrations from each environment as MP4 videos so you can visually verify the training data before investing hours in training. Shows one demo per environment with both cameras (agentview + wrist) side by side.

In [ ]:
# Generate MP4 videos of training demos (1 per environment, both cameras side-by-side)
import h5py
import numpy as np
import imageio
from pathlib import Path

DEMO_DIR = Path("data/robosuite_demos")
VIDEO_DIR = Path("outputs/videos/training_demos")
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

hdf5_files = sorted(DEMO_DIR.glob("*.hdf5"))
if not hdf5_files:
    print("No HDF5 demo files found — run Section 3 first.")
else:
    for hdf5_path in hdf5_files:
        env_name = hdf5_path.stem
        with h5py.File(hdf5_path, "r") as f:
            # Pick the first demo
            demo_key = sorted(f["data"].keys())[0]
            demo = f["data"][demo_key]

            # Load camera images
            agentview = demo["obs"]["agentview_image"][:]        # (T, H, W, 3)
            wrist = demo["obs"]["robot0_eye_in_hand_image"][:]   # (T, H, W, 3)

        n_frames = len(agentview)

        # Build side-by-side frames: [agentview | wrist]
        # Flip vertically (MuJoCo renders upside-down)
        frames = []
        for i in range(n_frames):
            left = np.flip(agentview[i], axis=0)
            right = np.flip(wrist[i], axis=0)
            # Add a 2px white separator between cameras
            sep = np.full((left.shape[0], 2, 3), 255, dtype=np.uint8)
            combined = np.concatenate([left, sep, right], axis=1)
            frames.append(combined)

        # Save MP4 at 20 fps (matches control frequency)
        video_path = VIDEO_DIR / f"{env_name}_demo.mp4"
        writer = imageio.get_writer(str(video_path), fps=20)
        for frame in frames:
            writer.append_data(frame)
        writer.close()

        print(f"  {env_name:25s} {n_frames:4d} frames -> {video_path}")

    print(f"\nTraining demo videos saved to {VIDEO_DIR}/")
    print(f"Total: {len(hdf5_files)} videos")

In [ ]:
# Watch training demo videos inline (one per environment)
import base64
from pathlib import Path
from IPython.display import HTML, display

VIDEO_DIR = Path("outputs/videos/training_demos")
mp4s = sorted(VIDEO_DIR.glob("*.mp4"))

if not mp4s:
    print("No training demo videos found — run the cell above first.")
else:
    print(f"Training demo videos ({len(mp4s)} environments)")
    print("Left = agentview camera | Right = wrist camera\n")

    for video_path in mp4s:
        env_name = video_path.stem.replace("_demo", "")
        with open(video_path, "rb") as f:
            data = base64.b64encode(f.read()).decode()
        display(HTML(
            f'<div style="display:inline-block; margin:8px; text-align:center">'
            f'<div style="font-weight:bold; font-size:14px; margin-bottom:4px">{env_name}</div>'
            f'<video controls autoplay loop muted width="400">'
            f'<source src="data:video/mp4;base64,{data}" type="video/mp4">'
            f'</video></div>'
        ))

## 4. Convert Demos to LeRobot v3.0 Format

Converts HDF5 demos using the **LeRobotDataset.create() API**, which automatically handles:
- Chunked parquet files with correct schema (`index`, `frame_index`, `episode_index`, `task_index`, `next.done`)
- MP4 video encoding for camera observations (`observation.images.image`, `observation.images.image2`)
- All metadata (`info.json`, `stats.json`, `tasks.jsonl`, `episodes/`)
- Normalization statistics for training

In [ ]:
!python data/convert_to_lerobot.py \
    --source robosuite \
    --hdf5-dir data/robosuite_demos \
    --output-dir outputs/lerobot/robosuite_multitask \
    --image-size 256

# Verify the dataset was created correctly
print("\n--- Verification ---")
import json
from pathlib import Path

info_path = Path("outputs/lerobot/robosuite_multitask/meta/info.json")
if info_path.exists():
    with open(info_path) as f:
        info = json.load(f)
    print(f"LeRobot v{info.get('codebase_version', '?')}")
    print(f"Episodes: {info.get('total_episodes', '?')}")
    print(f"Frames: {info.get('total_frames', '?')}")
    print(f"Tasks: {info.get('total_tasks', '?')}")
    print(f"FPS: {info.get('fps', '?')}")
    print(f"Features: {list(info.get('features', {}).keys())}")
else:
    print("ERROR: info.json not found — conversion may have failed")

# Check parquet files exist
parquets = list(Path("outputs/lerobot/robosuite_multitask/data").rglob("*.parquet"))
print(f"Parquet files: {len(parquets)}")

# Check tasks
tasks_path = Path("outputs/lerobot/robosuite_multitask/meta/tasks.jsonl")
if tasks_path.exists():
    with open(tasks_path) as f:
        tasks = [json.loads(line) for line in f]
    print(f"Tasks: {[t['task'] for t in tasks]}")

## 5. Train SmolVLA

Fine-tune SmolVLA on the robosuite multi-task dataset using `lerobot-train`.

**A100 required** — SmolVLA at batch_size=4 needs ~40GB VRAM.

**Important flags:**
- `--policy.repo_id` is **required** by LeRobot v0.4+ (will crash without it)
- `--policy.type=smolvla` auto-infers camera features from dataset (do NOT use `--policy.path`)
- `--use_amp=true` for mixed precision (do NOT pass `--policy.dtype` — crashes SmolVLA)

In [ ]:
# Training configuration
DATASET_ROOT = os.path.abspath("outputs/lerobot/robosuite_multitask")
OUTPUT_DIR = "outputs/checkpoints/smolvla_robosuite"
HF_USERNAME = "williampacini"  # Change to your HuggingFace username

BATCH_SIZE = 4        # SmolVLA is memory-heavy; official examples use 4
TOTAL_STEPS = 50000   # ~325 demos x ~250 steps = ~81k frames, ~2.5 epochs
SAVE_FREQ = 5000
EVAL_FREQ = 2000
SEED = 42

# For quick testing, uncomment:
# TOTAL_STEPS = 100
# SAVE_FREQ = 50
# EVAL_FREQ = 50

USE_WANDB = False  # Set True and run `wandb login` first
WANDB_PROJECT = "smolvla-robosuite"

print(f"Dataset: {DATASET_ROOT}")
print(f"Output: {OUTPUT_DIR}")
print(f"Steps: {TOTAL_STEPS}, Batch: {BATCH_SIZE}")

In [ ]:
# Build and run lerobot-train command
# NOTE: --use_amp=true for mixed precision, NOT --policy.dtype (crashes SmolVLA)
cmd = (
    f"lerobot-train"
    f" --policy.type=smolvla"
    f" --policy.load_vlm_weights=true"
    f" --policy.repo_id={HF_USERNAME}/smolvla-robosuite"
    f" --dataset.root={DATASET_ROOT}"
    f" --dataset.repo_id=robosuite_multitask"
    f" --batch_size={BATCH_SIZE}"
    f" --steps={TOTAL_STEPS}"
    f" --output_dir={OUTPUT_DIR}"
    f" --save_freq={SAVE_FREQ}"
    f" --eval_freq={EVAL_FREQ}"
    f" --seed={SEED}"
    f" --policy.device=cuda"
    f" --use_amp=true"
)

if USE_WANDB:
    cmd += f" --wandb.enable=true --wandb.project={WANDB_PROJECT}"

print("Training command:")
print(cmd)
print("\nStarting training...")
!{cmd}

In [ ]:
# Post-training fix: set n_action_steps = 50
# SmolVLA uses action chunking (50 actions per inference call)
import json
from pathlib import Path

OUTPUT_DIR = "outputs/checkpoints/smolvla_robosuite"  # redefine after restart

output_path = Path(OUTPUT_DIR)
if not output_path.exists():
    print(f"ERROR: {OUTPUT_DIR} does not exist. Training must complete first.")
else:
    config_files = list(output_path.rglob("config.json"))
    for config_path in config_files:
        with open(config_path) as f:
            cfg = json.load(f)
        old_val = cfg.get("n_action_steps", "not set")
        cfg["n_action_steps"] = 50
        with open(config_path, "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Fixed {config_path}: n_action_steps {old_val} -> 50")

    if not config_files:
        print(f"WARNING: No config.json found in {OUTPUT_DIR}")

## 6. Evaluate on Nut Assembly (Video of EVERY Episode)

Runs the trained model on **NutAssemblySingle** and **NutAssembly** tasks.
Records an MP4 video of every single episode so you can visually verify the results.

Videos are saved to `outputs/videos/robosuite/<env_name>/epNNN_success.mp4` or `epNNN_fail.mp4`.

In [ ]:
EVAL_EPISODES = 50   # episodes per task
# For quick testing, uncomment:
# EVAL_EPISODES = 2

OUTPUT_DIR = "outputs/checkpoints/smolvla_robosuite"  # redefine after restart

!python eval/eval_robosuite.py \
    --checkpoint {OUTPUT_DIR} \
    --nut-assembly \
    --episodes {EVAL_EPISODES} \
    --record-all \
    --output-dir outputs

In [ ]:
# Print results summary
import json
from pathlib import Path

results_file = Path("outputs/results/robosuite_eval.json")
if results_file.exists():
    with open(results_file) as f:
        results = json.load(f)

    print("=" * 60)
    print("  EVALUATION RESULTS")
    print("=" * 60)
    for t in results["tasks"]:
        print(f"  {t['env_name']:20s}  {t['success_rate']:.1%}  "
              f"({t['successes']}/{t['n_episodes']})  "
              f"mean_reward={t['mean_reward']:.2f}")
    print(f"  {'OVERALL':20s}  {results['overall_success_rate']:.1%}  "
          f"({results['total_successes']}/{results['total_trials']})")
    print()

    # Count videos
    for t in results["tasks"]:
        vdir = Path(t.get("video_dir", ""))
        if vdir.exists():
            mp4s = list(vdir.glob("*.mp4"))
            successes = [p for p in mp4s if "success" in p.name]
            print(f"  {t['env_name']}: {len(mp4s)} videos ({len(successes)} successes)")
else:
    print("No results file found. Run evaluation first.")

## 7. View Videos Inline

Display recorded episode videos directly in the notebook.

In [ ]:
import base64
from pathlib import Path
from IPython.display import HTML, display
import matplotlib.pyplot as plt
import imageio


def show_video(path, width=384):
    """Display MP4 video inline."""
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    tag = "SUCCESS" if "success" in Path(path).name else "FAIL"
    color = "green" if tag == "SUCCESS" else "red"
    display(HTML(
        f'<div style="display:inline-block; margin:4px; text-align:center">'
        f'<div style="color:{color}; font-weight:bold">{Path(path).stem} [{tag}]</div>'
        f'<video controls width="{width}">'
        f'<source src="data:video/mp4;base64,{data}" type="video/mp4">'
        f'</video></div>'
    ))


def show_thumbnail_grid(video_dir, cols=10):
    """Show first frame of each episode as a thumbnail grid."""
    mp4s = sorted(Path(video_dir).glob("*.mp4"))
    if not mp4s:
        print(f"No videos in {video_dir}")
        return

    rows = (len(mp4s) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.5))
    if rows == 1:
        axes = [axes]
    axes = [ax for row in axes for ax in (row if hasattr(row, '__iter__') else [row])]

    for i, ax in enumerate(axes):
        ax.axis("off")
        if i < len(mp4s):
            reader = imageio.get_reader(str(mp4s[i]))
            frame = reader.get_data(0)
            reader.close()
            is_success = "success" in mp4s[i].name
            border_color = "green" if is_success else "red"
            ax.imshow(frame)
            for spine in ax.spines.values():
                spine.set_edgecolor(border_color)
                spine.set_linewidth(3)
                spine.set_visible(True)
            ax.set_title(mp4s[i].stem.split("_")[0], fontsize=7)

    env_name = Path(video_dir).name
    n_success = sum(1 for p in mp4s if "success" in p.name)
    plt.suptitle(f"{env_name}: {n_success}/{len(mp4s)} success", fontsize=14)
    plt.tight_layout()
    plt.show()


print("Video display helpers loaded.")

In [ ]:
# Thumbnail grids — green border = success, red = fail
for env_name in ["NutAssemblySingle", "NutAssembly"]:
    video_dir = f"outputs/videos/robosuite/{env_name}"
    if Path(video_dir).exists():
        show_thumbnail_grid(video_dir)
    else:
        print(f"No videos found for {env_name}")

In [ ]:
# Watch individual episodes — show first 4 success + first 4 fail for each task
for env_name in ["NutAssemblySingle", "NutAssembly"]:
    video_dir = Path(f"outputs/videos/robosuite/{env_name}")
    if not video_dir.exists():
        continue

    print(f"\n{'='*50}")
    print(f"  {env_name}")
    print(f"{'='*50}")

    successes = sorted(video_dir.glob("*_success.mp4"))[:4]
    failures = sorted(video_dir.glob("*_fail.mp4"))[:4]

    if successes:
        print(f"\n  Successes ({len(successes)} shown):")
        for v in successes:
            show_video(v)

    if failures:
        print(f"\n  Failures ({len(failures)} shown):")
        for v in failures:
            show_video(v)

## 8. Save to Google Drive

Save checkpoint, videos, and results to Google Drive to survive Colab session timeouts.

In [ ]:
# Uncomment all lines below to save to Google Drive

# from google.colab import drive
# drive.mount('/content/drive')

# DRIVE_DIR = "/content/drive/MyDrive/AUTOLAB/robosuite"
# !mkdir -p {DRIVE_DIR}

# # Save checkpoint
# !cp -r outputs/checkpoints/smolvla_robosuite {DRIVE_DIR}/checkpoint/
# print("Checkpoint saved")

# # Save videos
# !cp -r outputs/videos/robosuite {DRIVE_DIR}/videos/
# print("Videos saved")

# # Save results
# !cp -r outputs/results {DRIVE_DIR}/results/
# print("Results saved")

# print(f"\nAll saved to {DRIVE_DIR}/")
# !ls -la {DRIVE_DIR}/

## Done!

**What we did:**
- Collected 325 scripted demos across 6 robosuite tasks
- **Previewed training demos as MP4 videos** (both cameras, side-by-side)
- Converted to LeRobot format
- Trained SmolVLA (50k steps on A100)
- Evaluated on NutAssemblySingle + NutAssembly
- Recorded MP4 video of **every** evaluation episode

**Files produced:**
- `outputs/videos/training_demos/` — one MP4 per environment showing expert demos (agentview + wrist)
- `outputs/checkpoints/smolvla_robosuite/` — trained model
- `outputs/videos/robosuite/NutAssemblySingle/` — one MP4 per eval episode
- `outputs/videos/robosuite/NutAssembly/` — one MP4 per eval episode
- `outputs/results/robosuite_eval.json` — success rates
- `outputs/results/robosuite_eval.csv` — summary table

**To evaluate all 6 tasks** (not just nut assembly), change `--nut-assembly` to `--all` in Section 6.